# CE49X – Linear Regression Challenge 1: Real Estate Valuation (Polynomial Regression + K-Fold CV)

Bu notebook, **Taiwan Real Estate Valuation** veri seti ile **polinomsal regresyon** (Polynomial Features + Linear Regression) kullanarak ev fiyatı tahmini yapmak için hazırlanmıştır.

Aynı modeli iki farklı şekilde değerlendiriyoruz:
- Farklı **train–test split** oranları ile *hold-out* değerlendirme
- Tüm veri üzerinde **5-fold cross-validation** değerlendirmesi

Ayrıca farklı **polinom derecelerini** (degree) deneyip hangisinin en iyi R² skorunu verdiğini karşılaştırıyoruz.

Adımlar:
1. Veri setini yükleme ve keşifsel veri analizi (EDA)
2. Özellik / hedef ayrımı ve veri ön işleme
3. Farklı train–test split oranları ile polinomsal regresyon modeli (degree = 1–5)
4. Her kombinasyon için R², MAE, RMSE hesaplama (hold-out)
5. 5-fold cross-validation ile aynı polinom derecelerini değerlendirme
6. Hold-out vs. K-Fold karşılaştırması ve **en iyi R²** değerine sahip modelin vurgulanması
7. En iyi derece ile tam veri üzerinde nihai model eğitimi
8. Katsayıların yorumu ve MRT mesafesinin etkisi
9. Verilen özelliklere sahip örnek bir konut için fiyat tahmini


## 1. Kütüphanelerin Yüklenmesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Grafik ayarları
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

RANDOM_STATE = 42  # Reprodüksiyon için sabit rastgelelik tohumu


## 2. Veri Setinin Yüklenmesi

Veri seti ödev açıklamasına göre `challenge/data/Real estate valuation data set.xlsx` yolunda bulunuyor. Aşağıda önce bu yolu varsayılan olarak kullanıyoruz; eğer notebook'u farklı bir klasörde çalıştırıyorsan `data_path` değişkenini güncellemen yeterli.


In [ ]:
# Veri dosya yolunu ayarla – gerekirse burayı kendi klasör yapına göre değiştir
data_path = Path('challenge/data/Real estate valuation data set.xlsx')

if not data_path.exists():
    # Alternatif: notebook ile aynı klasördeyse
    alt_path = Path('Real estate valuation data set.xlsx')
    if alt_path.exists():
        data_path = alt_path

print(f'Kullanılan veri yolu: {data_path}')

# Excel dosyasını yükle
df = pd.read_excel(data_path)

# Kolon isimlerindeki gereksiz boşlukları temizleyelim
df.columns = df.columns.str.strip()

df.head()

## 3. Keşifsel Veri Analizi (EDA)

Bu bölümde veri setinin boyutuna, veri tiplerine, eksik değerlere ve hedef değişkeninin temel istatistiklerine bakıyoruz.


In [ ]:
print('Veri seti boyutu:', df.shape)
print('\nVeri tipleri:')
print(df.dtypes)

print('\nEksik değer sayıları:')
print(df.isna().sum())

target_col = 'Y house price of unit area'
print('\nHedef değişken temel istatistikleri:')
print(df[target_col].describe())

In [ ]:
# Hedef değişken dağılımı
plt.hist(df[target_col], bins=20)
plt.title('Y house price of unit area dağılımı')
plt.xlabel('Fiyat (10,000 NTD / Ping)')
plt.ylabel('Frekans')
plt.show()

# Sayısal özellikler için korelasyon matrisi
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_title('Sayısal Değişkenler Korelasyon Matrisi')
fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)

# Eksen etiketleri
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

plt.tight_layout()
plt.show()

## 4. Özellik (X) ve Hedef (y) Ayrımı

- `No` kolonu yalnızca satır index'i işlevi görüyor, bu nedenle modelde kullanılmayacak.
- Özellikler: `X1`–`X6` sütunları
- Hedef: `Y house price of unit area`


In [ ]:
# 'No' kolonunu düşür (varsa)
if 'No' in df.columns:
    df = df.drop(columns=['No'])

feature_cols = [
    'X1 transaction date',
    'X2 house age',
    'X3 distance to the nearest MRT station',
    'X4 number of convenience stores',
    'X5 latitude',
    'X6 longitude'
]

X = df[feature_cols].copy()
y = df[target_col].copy()

print('Özellik matrisi boyutu:', X.shape)
print('Hedef vektör boyutu:', y.shape)
X.head()

## 5. Polinomsal Regresyon Modeli

Burada her derece için (degree = 1, 2, 3, 4, 5) aşağıdaki pipeline kullanılıyor:

1. `PolynomialFeatures(degree=d, include_bias=False)` – Orijinal özelliklerden polinom ve etkileşim terimleri üretir.
2. `LinearRegression()` – Bu yeni özellikler üzerinde klasik doğrusal regresyon uygular.

Degree = 1 olması durumunda, model basit doğrusal regresyon ile aynı etkiyi verir.


In [ ]:
def build_polynomial_model(degree: int) -> Pipeline:
    """Verilen derece için PolynomialFeatures + LinearRegression pipeline'ı döndürür."""
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('linreg', LinearRegression())
    ])
    return model


## 6. Farklı Train–Test Oranları ile Polinomsal Regresyon (Hold-Out)

Bu bölümde aşağıdaki kombinasyonlar için modeli eğitip değerlendiriyoruz:
- Polinom dereceleri: **degree = 1, 2, 3, 4, 5**
- Test oranları: **test_size = 0.2, 0.25, 0.3, 0.6**

Her kombinasyon için şu metrikleri hesaplıyoruz:
- Train R²
- Test R²
- Test MAE (Mean Absolute Error)
- Test RMSE (Root Mean Squared Error)


In [ ]:
degrees = [1, 2, 3, 4, 5]
test_sizes = [0.2, 0.25, 0.3, 0.6]

holdout_results = []

for degree in degrees:
    for test_size in test_sizes:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=RANDOM_STATE
        )

        model = build_polynomial_model(degree)
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        test_mae = mean_absolute_error(y_test, y_test_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

        holdout_results.append({
            'degree': degree,
            'test_size': test_size,
            'train_r2': train_r2,
            'test_r2': test_r2,
            'test_mae': test_mae,
            'test_rmse': test_rmse
        })

holdout_df = pd.DataFrame(holdout_results)

print('Hold-out sonuçları (degree & test_size kombinasyonları):')
holdout_df

In [ ]:
# Test R²'e göre en iyi hold-out sonucu
best_holdout = holdout_df.sort_values('test_r2', ascending=False).iloc[0]
print('En iyi HOLD-OUT sonucu:')
print(best_holdout)

# Görsel olarak da test R² karşılaştıralım
fig, ax = plt.subplots()
for degree in degrees:
    subset = holdout_df[holdout_df['degree'] == degree]
    ax.plot(subset['test_size'], subset['test_r2'], marker='o', label=f'Degree {degree}')

ax.set_title('Farklı Degree ve Test Oranları için Test R² (Hold-Out)')
ax.set_xlabel('Test oranı')
ax.set_ylabel('Test R²')
ax.legend(title='Degree')
plt.show()

## 7. 5-Fold Cross-Validation ile Polinomsal Regresyon

Bu bölümde, **K = 5** olacak şekilde 5-fold cross-validation uyguluyoruz.
Her degree için:
- 5 farklı fold üzerinde R² skorlarını hesaplıyoruz
- Ayrıca negatif MSE üzerinden RMSE'yi hesaplıyoruz

Sonuç olarak her degree için:
- Ortalama CV R² (cv_mean_r2)
- R² standart sapması (cv_std_r2)
- Ortalama CV RMSE (cv_mean_rmse)
- RMSE standart sapması (cv_std_rmse)


In [ ]:
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)

cv_results = []

for degree in degrees:
    model = build_polynomial_model(degree)

    # R² skorları (pozitif değer, yüksek olması iyi)
    r2_scores = cross_val_score(
        model, X, y,
        cv=kf,
        scoring='r2'
    )

    # Negatif MSE skorları (daha sonra RMSE'ye çevireceğiz)
    neg_mse_scores = cross_val_score(
        model, X, y,
        cv=kf,
        scoring='neg_mean_squared_error'
    )

    mse_scores = -neg_mse_scores
    rmse_scores = np.sqrt(mse_scores)

    cv_results.append({
        'degree': degree,
        'cv_mean_r2': r2_scores.mean(),
        'cv_std_r2': r2_scores.std(),
        'cv_mean_rmse': rmse_scores.mean(),
        'cv_std_rmse': rmse_scores.std()
    })

cv_df = pd.DataFrame(cv_results)

print('5-Fold Cross-Validation sonuçları:')
cv_df

In [ ]:
# CV R²'e göre en iyi derece
best_cv = cv_df.sort_values('cv_mean_r2', ascending=False).iloc[0]
print('En iyi K-FOLD (5-fold CV) sonucu:')
print(best_cv)

# Degree'lere göre ortalama CV R² grafiği
fig, ax = plt.subplots()
ax.plot(cv_df['degree'], cv_df['cv_mean_r2'], marker='o')
ax.set_title('Degree vs Ortalama CV R² (5-fold)')
ax.set_xlabel('Degree')
ax.set_ylabel('Ortalama CV R²')
plt.show()

# Degree'lere göre ortalama CV RMSE grafiği
fig, ax = plt.subplots()
ax.plot(cv_df['degree'], cv_df['cv_mean_rmse'], marker='o')
ax.set_title('Degree vs Ortalama CV RMSE (5-fold)')
ax.set_xlabel('Degree')
ax.set_ylabel('Ortalama CV RMSE')
plt.show()

## 8. Hold-Out vs. K-Fold Karşılaştırması ve En İyi Modelin Seçilmesi

Şimdi hem hold-out sonuçlarından hem de 5-fold CV sonuçlarından en yüksek R² skorlarını bulup hangisinin daha iyi olduğuna bakıyoruz.

- Hold-out için: `test_r2`
- K-Fold için: `cv_mean_r2`


In [ ]:
# En iyi hold-out ve en iyi CV sonucunu zaten hesapladık (best_holdout, best_cv)
print('--- ÖZET ---')
print('\nEn iyi HOLD-OUT sonucu:')
print(best_holdout)

print('\nEn iyi 5-FOLD CV sonucu:')
print(best_cv)

# Genel olarak en iyi R² hangisinde?
best_holdout_r2 = best_holdout['test_r2']
best_cv_r2 = best_cv['cv_mean_r2']

if best_holdout_r2 >= best_cv_r2:
    overall_best_method = 'hold-out'
    overall_best_degree = int(best_holdout['degree'])
    print('\nGenel olarak EN İYİ model: HOLD-OUT yöntemi ile')
    print(f"Degree = {overall_best_degree}, test_size = {best_holdout['test_size']}, Test R² = {best_holdout_r2:.4f}")
else:
    overall_best_method = 'kfold'
    overall_best_degree = int(best_cv['degree'])
    print('\nGenel olarak EN İYİ model: 5-FOLD CV yöntemi ile')
    print(f"Degree = {overall_best_degree}, Ortalama CV R² = {best_cv_r2:.4f}")


## 9. En İyi Derece ile Nihai Modelin Eğitilmesi

Yukarıdaki karşılaştırmaya göre **overall_best_degree** ile tam veri üzerinde nihai bir polinomsal regresyon modeli eğitiyoruz. Bu model daha sonra katsayı yorumu ve örnek ev fiyatı tahmini için kullanılacak.


In [ ]:
# overall_best_degree değişkeni bir önceki hücrede belirlenmiş olmalı
final_degree = overall_best_degree
print(f'Nihai model için seçilen degree: {final_degree}')

final_model = build_polynomial_model(final_degree)
final_model.fit(X, y)

print('Nihai model, tüm veri üzerinde eğitildi.')

## 10. Katsayıların Analizi (Polinomsal Model)

Polinomsal modelde, orijinal her özelliğin birden fazla türevi (kareleri, küpleri, etkileşim terimleri vb.) olduğu için katsayı sayısı artar.

Aşağıda:
- `PolynomialFeatures` tarafından üretilen tüm özellik isimlerini alıyoruz.
- Her özellik için katsayıyı hesaplayıp mutlak değerine göre sıralıyoruz.
- En büyük katsayılara sahip ilk 10 özelliği listeliyoruz.

Ayrıca MRT mesafesi (`X3 distance to the nearest MRT station`) içeren terimlere özellikle bakıyoruz.


In [ ]:
# Pipeline içindeki adımlara erişim
poly_step = final_model.named_steps['poly']
linreg_step = final_model.named_steps['linreg']

# Polinomsal özellik isimleri
poly_feature_names = poly_step.get_feature_names_out(feature_cols)
coefficients = linreg_step.coef_

coef_df = pd.DataFrame({
    'feature': poly_feature_names,
    'coefficient': coefficients
})
coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
coef_df_sorted = coef_df.sort_values('abs_coefficient', ascending=False)

print('En büyük katsayıya sahip ilk 10 polinomsal özellik:')
coef_df_sorted.head(10)

In [ ]:
# MRT mesafesini içeren terimlere özel olarak bakalım
mrt_mask = coef_df['feature'].str.contains('X3 distance to the nearest MRT station')
mrt_terms = coef_df[mrt_mask].sort_values('abs_coefficient', ascending=False)
print("\nMRT mesafesi ile ilgili polinomsal terimler ve katsayıları:")
mrt_terms.head(10)

### MRT Mesafesi Katsayılarının Yorumu

- Eğer **MRT mesafesini içeren temel terim** (örneğin `X3 distance to the nearest MRT station`) veya onun daha düşük dereceli polinomları için katsayılar genellikle **negatif** ise:
  - MRT istasyonuna olan mesafe arttıkça (ev metrodan uzaklaştıkça) fiyatın azaldığını gösterir.
  - Bu, ulaşım erişilebilirliğinin konut fiyatı üzerinde pozitif bir etkisi olduğu şeklinde yorumlanabilir (metroya yakın evler daha pahalı).
- Polinomsal modelde aynı değişkeni içeren birçok terim olduğu için yorumu yaparken genel örüntüye bakmak gerekir: katsayıların işareti ve büyüklüğü hep aynı yönde mi?


## 11. Örnek Bir Konut İçin Fiyat Tahmini

Ödev tanımında verilen özelliklere sahip bir konut için fiyat tahmini yapalım:
- House age = **5 yıl**
- Distance to MRT = **500 metre**
- Number of convenience stores = **3**
- Transaction date = **2013.5**
- Latitude ve longitude = veri setindeki **medyan** değerler


In [ ]:
# Latitude ve longitude için medyan değerler
lat_median = X['X5 latitude'].median()
lon_median = X['X6 longitude'].median()

print('Medyan enlem (latitude):', lat_median)
print('Medyan boylam (longitude):', lon_median)

# Verilen özelliklere sahip örnek konut
example_house = pd.DataFrame({
    'X1 transaction date': [2013.5],
    'X2 house age': [5.0],
    'X3 distance to the nearest MRT station': [500.0],
    'X4 number of convenience stores': [3],
    'X5 latitude': [lat_median],
    'X6 longitude': [lon_median]
})

predicted_price = final_model.predict(example_house)[0]
print(f'Tahmin edilen fiyat (Y house price of unit area): {predicted_price:.2f} (10,000 NTD / Ping)')

## 12. Kısa Özet

Bu notebookta şunları yaptık:
1. Real Estate Valuation veri setini yükleyip inceledik.
2. X1–X6 özelliklerini ve `Y house price of unit area` hedefini ayırdık.
3. Polinomsal regresyon (PolynomialFeatures + LinearRegression) ile degree = 1–5 için farklı train–test split oranlarını denedik (0.2, 0.25, 0.3, 0.6) ve hold-out metriklerini hesapladık.
4. Aynı dereceler için 5-fold cross-validation uygulayıp ortalama R² ve RMSE değerlerini çıkardık.
5. Hold-out ve K-Fold sonuçlarını karşılaştırarak **en yüksek R²** değerine sahip modeli vurguladık.
6. Bu en iyi degree ile tam veri üzerinde nihai modeli eğittik ve polinomsal katsayıları analiz ettik.
7. Son olarak, verilen özelliklere sahip örnek bir konut için fiyat tahmini yaptık.

Bu notebook'u kullanarak raporunda hem:
- Farklı split oranlarına göre performans karşılaştırması,
- Hem de K-Fold cross-validation sonuçlarını

grafik ve tablolarla rahatça anlatabilirsin. İstersen ridge/lasso gibi düzenlileştirilmiş modelleri de aynı şablonla ekleyebilirsin.
